In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

# =========================
# 1. LOAD DATASET
# =========================
train_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/train',
    label_mode='int',
    image_size=(224,224),
    batch_size=32,
    shuffle=True,
    color_mode='rgb'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/val',
    label_mode='int',
    image_size=(224,224),
    batch_size=32,
    shuffle=False,
    color_mode='rgb'
)

num_classes = len(train_ds.class_names)

# =========================
# 2. YOUR LABEL DICT (PASTE FULL HERE)
# =========================
label_dict = {
    0: 'અ', 1: 'આ', 2: 'ઇ', 3: 'ઈ', 4: 'ઉ', 5: 'ઊ', 6: 'ઋ', 7: 'એ', 8: 'ઐ', 9: 'ઓ', 10: 'ઔ', 11: 'અં',
    12: 'ક', 13: 'કા', 14: 'કિ', 15: 'કી', 16: 'કુ', 17: 'કૂ', 18: 'કે', 19: 'કૈ', 20: 'કો', 21: 'કૌ', 22: 'કં', 23: 'કઃ',
    24: 'ખ', 25: 'ખા', 26: 'ખિ', 27: 'ખી', 28: 'ખુ', 29: 'ખૂ', 30: 'ખે', 31: 'ખૈ', 32: 'ખો', 33: 'ખૌ', 34: 'ખં', 35: 'ખઃ',
    36: 'ગ', 37: 'ગા', 38: 'ગિ', 39: 'ગી', 40: 'ગુ', 41: 'ગૂ', 42: 'ગે', 43: 'ગૈ', 44: 'ગો', 45: 'ગૌ', 46: 'ગં', 47: 'ગઃ',
    48: 'ઘ', 49: 'ઘા', 50: 'ઘિ', 51: 'ઘી', 52: 'ઘુ', 53: 'ઘૂ', 54: 'ઘે', 55: 'ઘૈ', 56: 'ઘો', 57: 'ઘૌ', 58: 'ઘં', 59: 'ઘઃ',
    60: 'ચ', 61: 'ચા', 62: 'ચિ', 63: 'ચી', 64: 'ચુ', 65: 'ચૂ', 66: 'ચે', 67: 'ચૈ', 68: 'ચો', 69: 'ચૌ', 70: 'ચં', 71: 'ચઃ',
    72: 'છ', 73: 'છા', 74: 'છિ', 75: 'છી', 76: 'છુ', 77: 'છૂ', 78: 'છે', 79: 'છૈ', 80: 'છો', 81: 'છૌ', 82: 'છં', 83: 'છઃ',
    84: 'જ', 85: 'જા', 86: 'જિ', 87: 'જી', 88: 'જુ', 89: 'જૂ', 90: 'જે', 91: 'જૈ', 92: 'જો', 93: 'જૌ', 94: 'જં', 95: 'જઃ',
    96: 'ઝ', 97: 'ઝા', 98: 'ઝિ', 99: 'ઝી', 100: 'ઝુ', 101: 'ઝૂ', 102: 'ઝે', 103: 'ઝૈ', 104: 'ઝો', 105: 'ઝૌ', 106: 'ઝં', 107: 'ઝઃ',
    108: 'ટ', 109: 'ટા', 110: 'ટિ', 111: 'ટી', 112: 'ટુ', 113: 'ટૂ', 114: 'ટે', 115: 'ટૈ', 116: 'ટો', 117: 'ટૌ', 118: 'ટં', 119: 'ટઃ',
    120: 'ઠ', 121: 'ઠા', 122: 'ઠિ', 123: 'ઠી', 124: 'ઠુ', 125: 'ઠૂ', 126: 'ઠે', 127: 'ઠૈ', 128: 'ઠો', 129: 'ઠૌ', 130: 'ઠં', 131: 'ઠઃ',
    132: 'ડ', 133: 'ડા', 134: 'ડિ', 135: 'ડી', 136: 'ડુ', 137: 'ડૂ', 138: 'ડે', 139: 'ડૈ', 140: 'ડો', 141: 'ડૌ', 142: 'ડં', 143: 'ડઃ',
    144: 'ઢ', 145: 'ઢા', 146: 'ઢિ', 147: 'ઢી', 148: 'ઢુ', 149: 'ઢૂ', 150: 'ઢે', 151: 'ઢૈ', 152: 'ઢો', 153: 'ઢૌ', 154: 'ઢં', 155: 'ઢઃ',
    156: 'ણ', 157: 'ણા', 158: 'ણિ', 159: 'ણી', 160: 'ણુ', 161: 'ણૂ', 162: 'ણે', 163: 'ણૈ', 164: 'ણો', 165: 'ણૌ', 166: 'ણં', 167: 'ણઃ',
    168: 'ત', 169: 'તા', 170: 'તિ', 171: 'તી', 172: 'તુ', 173: 'તૂ', 174: 'તે', 175: 'તૈ', 176: 'તો', 177: 'તૌ', 178: 'તં', 179: 'તઃ',
    180: 'થ', 181: 'થા', 182: 'થિ', 183: 'થી', 184: 'થુ', 185: 'થૂ', 186: 'થે', 187: 'થૈ', 188: 'થો', 189: 'થૌ', 190: 'થં', 191: 'થઃ',
    192: 'દ', 193: 'દા', 194: 'દિ', 195: 'દી', 196: 'દુ', 197: 'દૂ', 198: 'દે', 199: 'દૈ', 200: 'દો', 201: 'દૌ', 202: 'દં', 203: 'દઃ',
    204: 'ધ', 205: 'ધા', 206: 'ધિ', 207: 'ધી', 208: 'ધુ', 209: 'ધૂ', 210: 'ધે', 211: 'ધૈ', 212: 'ધો', 213: 'ધૌ', 214: 'ધં', 215: 'ધઃ',
    216: 'ન', 217: 'ના', 218: 'નિ', 219: 'ની', 220: 'નુ', 221: 'નૂ', 222: 'ને', 223: 'નૈ', 224: 'નો', 225: 'નૌ', 226: 'નં', 227: 'નઃ',
    228: 'પ', 229: 'પા', 230: 'પિ', 231: 'પી', 232: 'પુ', 233: 'પૂ', 234: 'પે', 235: 'પૈ', 236: 'પો', 237: 'પૌ', 238: 'પં', 239: 'પઃ',
    240: 'ફ', 241: 'ફા', 242: 'ફિ', 243: 'ફી', 244: 'ફુ', 245: 'ફૂ', 246: 'ફે', 247: 'ફૈ', 248: 'ફો', 249: 'ફૌ', 250: 'ફં', 251: 'ફઃ',
    252: 'બ', 253: 'બા', 254: 'બિ', 255: 'બી', 256: 'બુ', 257: 'બૂ', 258: 'બે', 259: 'બૈ', 260: 'બો', 261: 'બૌ', 262: 'બં', 263: 'બઃ',
    264: 'ભ', 265: 'ભા', 266: 'ભિ', 267: 'ભી', 268: 'ભુ', 269: 'ભૂ', 270: 'ભે', 271: 'ભૈ', 272: 'ભો', 273: 'ભૌ', 274: 'ભં', 275: 'ભઃ',
    276: 'મ', 277: 'મા', 278: 'મિ', 279: 'મી', 280: 'મુ', 281: 'મૂ', 282: 'મે', 283: 'મૈ', 284: 'મો', 285: 'મૌ', 286: 'મં', 287: 'મઃ',
    288: 'ય', 289: 'યા', 290: 'યિ', 291: 'યી', 292: 'યુ', 293: 'યૂ', 294: 'યે', 295: 'યૈ', 296: 'યો', 297: 'યૌ', 298: 'યં', 299: 'યઃ',
    300: 'ર', 301: 'રા', 302: 'રિ', 303: 'રી', 304: 'રુ', 305: 'રૂ', 306: 'રે', 307: 'રૈ', 308: 'રો', 309: 'રૌ', 310: 'રં', 311: 'રઃ',
    312: 'લ', 313: 'લા', 314: 'લિ', 315: 'લી', 316: 'લુ', 317: 'લૂ', 318: 'લે', 319: 'લૈ', 320: 'લો', 321: 'લૌ', 322: 'લં', 323: 'લઃ',
    324: 'વ', 325: 'વા', 326: 'વિ', 327: 'વી', 328: 'વુ', 329: 'વૂ', 330: 'વે', 331: 'વૈ', 332: 'વો', 333: 'વૌ', 334: 'વં', 335: 'વઃ',
    336: 'શ', 337: 'શા', 338: 'શિ', 339: 'શી', 340: 'શુ', 341: 'શૂ', 342: 'શે', 343: 'શૈ', 344: 'શો', 345: 'શૌ', 346: 'શં', 347: 'શઃ',
    348: 'ષ', 349: 'ષા', 350: 'ષિ', 351: 'ષી', 352: 'ષુ', 353: 'ષૂ', 354: 'ષે', 355: 'ષૈ', 356: 'ષો', 357: 'ષૌ', 358: 'ષં', 359: 'ષઃ',
    360: 'સ', 361: 'સા', 362: 'સિ', 363: 'સી', 364: 'સુ', 365: 'સૂ', 366: 'સે', 367: 'સૈ', 368: 'સો', 369: 'સૌ', 370: 'સં', 371: 'સઃ',
    372: 'હ', 373: 'હા', 374: 'હિ', 375: 'હી', 376: 'હુ', 377: 'હૂ', 378: 'હે', 379: 'હૈ', 380: 'હો', 381: 'હૌ', 382: 'હં', 383: 'હઃ',
    384: 'ળ', 385: 'ળા', 386: 'ળિ', 387: 'ળી', 388: 'ળુ', 389: 'ળૂ', 390: 'ળે', 391: 'ળૈ', 392: 'ળો', 393: 'ળૌ', 394: 'ળં', 395: 'ળઃ',
    396: 'ક્ષ', 397: 'ક્ષા', 398: 'ક્ષિ', 399: 'ક્ષી', 400: 'ક્ષુ', 401: 'ક્ષૂ', 402: 'ક્ષે', 403: 'ક્ષૈ', 404: 'ક્ષો', 405: 'ક્ષૌ', 406: 'ક્ષં', 407: 'ક્ષઃ',
    408: 'ત્ર', 409: 'ત્રા', 410: 'ત્રિ', 411: 'ત્રી', 412: 'ત્રુ', 413: 'ત્રૂ', 414: 'ત્રે', 415: 'ત્રૈ', 416: 'ત્રો', 417: 'ત્રૌ', 418: 'ત્રં', 419: 'ત્રઃ',
    420: 'જ્ઞ', 421: 'જ્ઞા', 422: 'જ્ઞિ', 423: 'જ્ઞી', 424: 'જ્ઞુ', 425: 'જ્ઞૂ', 426: 'જ્ઞે', 427: 'જ્ઞૈ', 428: 'જ્ઞો', 429: 'જ્ઞૌ', 430: 'જ્ઞં', 431: 'જ્ઞઃ'
}

# =========================
# 3. BASES + MATRAS (MANUAL SAFE)
# =========================
bases = [
    'અ','આ','ઇ','ઈ','ઉ','ઊ','ઋ','એ','ઐ','ઓ','ઔ','અં',
    'ક','ખ','ગ','ઘ','ચ','છ','જ','ઝ',
    'ટ','ઠ','ડ','ઢ','ણ',
    'ત','થ','દ','ધ','ન',
    'પ','ફ','બ','ભ','મ',
    'ય','ર','લ','વ',
    'શ','ષ','સ','હ','ળ',
    'ક્ષ','ત્ર','જ્ઞ'
]

matras = [
    '', 'ા','િ','ી','ુ','ૂ',
    'ે','ૈ','ો','ૌ',
    'ં','ઃ'
]

base_to_idx = {b:i for i,b in enumerate(bases)}
matra_to_idx = {m:i for i,m in enumerate(matras)}

# =========================
# 4. SPLIT FUNCTION
# =========================
def split_char(char):
    if char.startswith('ક્ષ'):
        base = 'ક્ષ'
        rest = char[2:]
    elif char.startswith('ત્ર'):
        base = 'ત્ર'
        rest = char[2:]
    elif char.startswith('જ્ઞ'):
        base = 'જ્ઞ'
        rest = char[2:]
    else:
        base = char[0]
        rest = char[1:]

    if rest in matras:
        matra = rest
    else:
        matra = ''

    return base, matra

# =========================
# 5. CREATE LOOKUPS (CRITICAL)
# =========================
base_lookup = np.zeros(num_classes, dtype=np.int32)
matra_lookup = np.zeros(num_classes, dtype=np.int32)

for i in range(num_classes):
    char = label_dict[i]   # SAFE: using index directly

    base, matra = split_char(char)

    if base not in base_to_idx:
        raise ValueError(f"Missing base: {base}")

    if matra not in matra_to_idx:
        raise ValueError(f"Missing matra: {matra}")

    base_lookup[i] = base_to_idx[base]
    matra_lookup[i] = matra_to_idx[matra]

base_lookup = tf.constant(base_lookup)
matra_lookup = tf.constant(matra_lookup)

print("Base classes:", len(bases))
print("Matra classes:", len(matras))

# =========================
# 6. MAP FUNCTION (FIXED)
# =========================
def map_fn(x, y):
    base = tf.gather(base_lookup, y)
    matra = tf.gather(matra_lookup, y)

    return x, {
        'base': base,
        'matra': matra
    }

train_ds = train_ds.map(map_fn)
test_ds = test_ds.map(map_fn)

# =========================
# 7. NORMALIZE
# =========================
train_ds = train_ds.map(lambda x,y: (tf.cast(x, tf.float32)/255.0, y))
test_ds = test_ds.map(lambda x,y: (tf.cast(x, tf.float32)/255.0, y))

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

# =========================
# 8. MODEL
# =========================
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224,224,3)
)

base_model.trainable = False

# 👉 OPTIONAL (better performance)
for layer in base_model.layers[-20:]:
    layer.trainable = True

inputs = layers.Input(shape=(224,224,3))

x = preprocess_input(inputs)
x = base_model(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.4)(x)

base_output = layers.Dense(len(bases), activation='softmax', name='base')(x)
matra_output = layers.Dense(len(matras), activation='softmax', name='matra')(x)

model = Model(inputs, [base_output, matra_output])

# =========================
# 9. COMPILE
# =========================
model.compile(
    optimizer='adam',
    loss={
        'base': 'sparse_categorical_crossentropy',
        'matra': 'sparse_categorical_crossentropy'
    },
    metrics={
        'base': 'accuracy',
        'matra': 'accuracy'
    }
)

model.summary()

# =========================
# 10. TRAIN
# =========================
model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20
)

# =========================
# 11. PREDICTION
# =========================
idx_to_base = {v:k for k,v in base_to_idx.items()}
idx_to_matra = {v:k for k,v in matra_to_idx.items()}

for images, labels in test_ds.take(1):
    base_pred, matra_pred = model.predict(images)

    base_ids = tf.argmax(base_pred, axis=1).numpy()
    matra_ids = tf.argmax(matra_pred, axis=1).numpy()

    for i in range(10):
        pred_char = idx_to_base[base_ids[i]] + idx_to_matra[matra_ids[i]]
        print("Pred:", pred_char)

Found 2484 files belonging to 432 classes.
Found 860 files belonging to 432 classes.
Base classes: 47
Matra classes: 12


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ efficientnetb0 (Functional)   │ (None, 7, 7, 1280)        │       4,049,571 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling2d      │ (None, 1280)              │               0 │ efficientnetb0[0][0]       │
│ (GlobalAveragePooling2D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 1280)              │           5,120 │ global_average_pooling2d[… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 512)               │         655,872 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 512)               │               0 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ base (Dense)                  │ (None, 47)                │          24,111 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ matra (Dense)                 │ (None, 12)                │           6,156 │ dropout[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,740,830 (18.08 MB)

 Trainable params: 2,039,659 (7.78 MB)

 Non-trainable params: 2,701,171 (10.30 MB)

Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 75s 833ms/step - base_accuracy: 0.0310 - base_loss: 4.7665 - loss: 8.1625 - matra_accuracy: 0.1047 - matra_loss: 3.3902 - val_base_accuracy: 0.0291 - val_base_loss: 3.7801 - val_loss: 6.2833 - val_matra_accuracy: 0.1814 - val_matra_loss: 2.5006
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 65s 834ms/step - base_accuracy: 0.0238 - base_loss: 4.5090 - loss: 7.6381 - matra_accuracy: 0.1067 - matra_loss: 3.1273 - val_base_accuracy: 0.0279 - val_base_loss: 3.7466 - val_loss: 6.3947 - val_matra_accuracy: 0.0733 - val_matra_loss: 2.6461
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 77s 771ms/step - base_accuracy: 0.0306 - base_loss: 4.1650 - loss: 6.9283 - matra_accuracy: 0.1244 - matra_loss: 2.7632 - val_base_accuracy: 0.0244 - val_base_loss: 3.7401 - val_loss: 6.2880 - val_matra_accuracy: 0.0767 - val_matra_loss: 2.5434
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 90s 870ms/step - base_accuracy: 0.0334 - base_loss: 3.8724 - loss: 6.3837 - matra_accuracy: 0.1457 - matra_loss: 